# FIT5196 Assessment 1 - EDA

---



#### Group64:
Member 1: Hao Xie (32613571), hxie0035@student.monash.edu, Contribution

Member 2: Yilin Shen, (34754954), yshe0132@student.monash.edu, EDA Visualization

...

---

### Table of Content

1. Load, parse and merge data files
2. EDA
3. Key insights and research
1.   List item
2.   List item





# 1. Load, parse and merge data files

In [ ]:
# Import all the modules
import json
import re

import pandas as pd
import numpy as np

1. Load and Parse the XML file

In [ ]:
# pd.read_xmlparses the xml into dataframe directly
df_xml = pd.read_xml('group_64.xml', encoding='utf-8')
print("XML shape:", df_xml.shape)
print(df_xml.columns.tolist())

2. Load and Parse the JSON File

In [ ]:
# json.load() reads the file into a Python list of dicts; pd.DataFrame converts it
with open('group_64.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_json = pd.DataFrame(data)
print("JSON shape:", df_json.shape)
print(df_json.head())

3. Schema Alignment - Rename Columns and Enforce Data Types

In [ ]:
# Rename JSON columns so they match the XML naming convention (underscores)
df_json = df_json.rename(columns = {
    'Post date': 'Post_date',
    'Taken date': 'Taken_date'
})
dtype_mapping = {
    'PostID': 'Int64',
    'UserID': 'string',
    'secret': 'string',
    'server': 'Int64',
    'title': 'string',
    'ispublic': 'Int64',
    'isfriend': 'Int64',
    'isfamily': 'Int64',
    'farm': 'Int64',
    'City': 'string',
    'Country': 'string',
    'tags': 'string',
    'description': 'string'
}
# Apply type casting to both DataFrames
datetime_cols = ['Post_date', 'Taken_date', 'min_taken_date']
# Convert date coulmns to proper datetime objects
df_json = df_json.astype(dtype_mapping)
df_xml = df_xml.astype(dtype_mapping)
for col in datetime_cols:
    df_json[col] = pd.to_datetime(df_json[col])
    df_xml[col] = pd.to_datetime(df_xml[col])

4. Merge (Concatenate) the Two Datasets

In [ ]:

# Merge (Concatenate) the Two Datasets
df_merged = pd.concat([df_xml, df_json], ignore_index = True)
print(df_merged.shape)

5. Duplication

In [ ]:

# 5a. Remove duplicate rows across all columns
df_merged = df_merged.drop_duplicates()
print(df_merged.shape)



In [ ]:
# Inspect remaining PostID-level duplicates
dup_count = df_merged['PostID'].duplicated().sum()
print(dup_count)


Result show 29 duplicate rows remaining after drop duplicates

In [ ]:
# Investigation: duplicates differ only in min_taken_date
duplicates = df_merged[df_merged['PostID'].duplicated(keep=False)]
sample_id = duplicates['PostID'].iloc[0]
print(df_merged[df_merged['PostID'] == sample_id])

Output shows the two identical rows only differs in min_taken_date, we can resolve by keeping the record with the earliest min_taken_date

In [ ]:
df_merged = df_merged.sort_values('min_taken_date', ascending=True)
df_merged = df_merged.drop_duplicates(subset=['PostID'], keep='first')
df_merged = df_merged.reset_index(drop=True)
print(df_merged.shape)



Verify: no remaining PostID duplicates

In [ ]:
assert df_merged['PostID'].duplicated().sum() == 0, "PostID duplicates still exist!"

6. Text Cleaning (Regular Expressions - applied only to text columns)

In [ ]:
# Define the target text columns (as required by the spec)
text_cols = ['title', 'City', 'Country', 'tags', 'description']


Define  regex patterns for efficiency (compiled first and reuse it later)

In [ ]:
# XML/HTML pattern: Marches tags like <br>, <p class="x", </div>, dtc.
RE_HTML_TAGS = re.compile(r'<[^>]+>')

In [ ]:
# Match emojis, pictographs, zero-width joiners, and variation selectors.
RE_EMOJI = re.compile(
    '['
    '\U0001F600-\U0001F64F'   # Emoticons (smileys, gestures)
    '\U0001F300-\U0001F5FF'   # Miscellaneous Symbols and Pictographs
    '\U0001F680-\U0001F6FF'   # Transport and Map Symbols
    '\U0001F1E0-\U0001F1FF'   # Regional Indicator Symbols (flags)
    '\U0001F900-\U0001F9FF'   # Supplemental Symbols and Pictographs
    '\U0001FA00-\U0001FA6F'   # Chess Symbols
    '\U0001FA70-\U0001FAFF'   # Symbols and Pictographs Extended-A
    '\U00002702-\U000027B0'   # Dingbats
    '\U00002600-\U000026FF'   # Miscellaneous Symbols
    '\U0000FE00-\U0000FE0F'   # Variation Selectors
    '\U0000200D'              # Zero Width Joiner
    '\U00002300-\U000023FF'   # Miscellaneous Technical (some emoji)
    '\U0000200C-\U0000200D'   # Zero Width Non-Joiner + Joiner
    '\U000000A9'              # Copyright symbol (sometimes rendered as emoji)
    '\U000000AE'              # Registered symbol
    '\U0000203C'              # Double exclamation mark
    '\U00002049'              # Exclamation question mark
    '\U000020E3'              # Combining Enclosing Keycap
    '\U00002122'              # Trade mark sign
    '\U00002139'              # Information source
    '\U00002194-\U000021AA'   # Arrows (sometimes emoji)
    '\U000025AA-\U000025AB'   # Small squares
    '\U000025B6'              # Play button
    '\U000025C0'              # Reverse button
    '\U000025FB-\U000025FE'   # Medium squares
    '\U00003030'              # Wavy dash
    '\U0000303D'              # Part alternation mark
    '\U00003297'              # Circled ideograph congratulation
    '\U00003299'              # Circled ideograph secret
    ']+',
    flags=re.UNICODE
)

In [ ]:
# non-Latin script Pattern
# Allowed: Latin script (Basic Latin, Latin-1 Supplement, Latin Extended-A/B,
#          Latin Extended Additional), combining diacritical marks,
#          digits, whitespace, ASCII punctuation, common Unicode punctuation, and @.

RE_NON_LATIN = re.compile(
    r'[^'
    r'a-zA-Z'                 # Basic Latin letters
    r'\u00C0-\u00D6'          # Latin-1 Supplement (upper accented: A-grave to O-diaeresis)
    r'\u00D8-\u00F6'          # Latin-1 Supplement (O-stroke to o-diaeresis)
    r'\u00F8-\u00FF'          # Latin-1 Supplement (o-stroke to y-diaeresis)
    r'\u0100-\u017F'          # Latin Extended-A (e.g., macrons, carons)
    r'\u0180-\u024F'          # Latin Extended-B (e.g., Croatian, Romanian digraphs)
    r'\u1E00-\u1EFF'          # Latin Extended Additional (e.g., Vietnamese)
    r'\u0300-\u036F'          # Combining Diacritical Marks (accents, tildes, etc.)
    r'\s'                     # Whitespace (space, tab, newline)
    r'0-9'                    # Digits
    r'!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~'   # ASCII punctuation
    r'\u2010-\u2027'          # General Punctuation (en-dash, em-dash, quotes, etc.)
    r'\u2030-\u205E'          # Per-mille, prime, brackets, etc.
    r']'
)

In [ ]:
def clean_text(text):
    """
    Clean a single text value by sequentially removing:
      1. XML/HTML tags (e.g., <br>, <a href="...">)
      2. Emojis and pictographic symbols
      3. Non-Latin characters (Japanese, Chinese, Russian Cyrillic, etc.)
    Retains: Latin script (including European diacritics), digits, whitespace,
             punctuation, and the @ symbol.
    Returns the original value unchanged if it is not a string (e.g., NaN).
    """
    if not isinstance(text, str):
        return text
    # Step 1: Remove XML/HTML tags
    text = RE_HTML_TAGS.sub('', text)
    # Step 2: Remove emojis and associated joiners / variation selectors
    text = RE_EMOJI.sub('', text)
    # Step 3: Remove non-Latin characters
    text = RE_NON_LATIN.sub('', text)
    return text

# Apply cleaning to all string/object columns
string_cols = df_merged.select_dtypes(include=['string', 'object']).columns.tolist()
for col in string_cols:
    df_merged[col] = df_merged[col].map(clean_text)

7. Lowercase Transformation

In [ ]:
# .str.lower() automatically preserves NaN values
df_merged[text_cols] = df_merged[text_cols].apply(lambda x: x.str.lower())
print(df_merged[text_cols].head(10))

8. Date Formatting - Standardise min_taken_date Display

In [ ]:
# Standardise min_taken_date to a consistent string for output
df_merged['min_taken_date'] = (
    pd.to_datetime(df_merged['min_taken_date'])
    .dt.strftime('%Y-%m-%d %H:%M:%S')
)
# Convert min_taken_date back to datetime type
df_merged['min_taken_date'] = pd.to_datetime(df_merged['min_taken_date'])
print(df_merged['min_taken_date'])
print(df_merged.dtypes)

9. Null Value Handling

In [ ]:
# Replace empty strings and whitespace-only strings with NaN
df_merged = df_merged.replace(r'^\s*$', np.nan, regex=True)

In [ ]:
# Replace common null-like strings with NaN
null_like_values = ['none', 'null', 'na', 'n/a', 'None', 'NULL', 'NA', 'N/A', '']
df_merged = df_merged.replace(null_like_values, np.nan)
print(df_merged.isna().sum())

10. Validation Checks (tageted to text columns)

In [ ]:
# 10a. Check for remaining XML/HTML tags
def has_html_tags(text):
    if not isinstance(text, str):
        return False
    return bool(RE_HTML_TAGS.search(text))

In [ ]:
html_counts = df_merged[string_cols].apply(lambda col: col.map(has_html_tags)).sum()
print("XML/HTML tags remaining:")
print(html_counts)
assert html_counts.sum() == 0, "XML/HTML tags still present!"

In [ ]:
# 10b. Check for remaining non-Latin characters
def has_non_latin(text):
    if not isinstance(text, str):
        return False
    return bool(RE_NON_LATIN.search(text))

non_latin_counts = df_merged[string_cols].apply(lambda col: col.map(has_non_latin)).sum()
print("Non-Latin characters remaining:")
print(non_latin_counts)
assert non_latin_counts.sum() == 0, "Non-Latin characters still present!"

In [ ]:
# 10c. Check for remaining emojis
def has_emoji(text):
    if not isinstance(text, str):
        return False
    return bool(RE_EMOJI.search(text))

emoji_counts = df_merged[string_cols].apply(lambda col: col.map(has_emoji)).sum()
print(f"Emoji characters remaining:\n{emoji_counts}")
assert emoji_counts.sum() == 0, "Emoji characters still present!"

In [ ]:
# 10d. Check no null-like string literals remain
null_like_check = ['None', 'none', 'NULL', 'null', 'NA', 'na',
                   'N/A', 'n/a', 'nan', 'NaN', '']
for col in df_merged.columns:
    matches = df_merged[col].isin(null_like_check).sum()
    if matches > 0:
        print(f"  Warning: Column '{col}' has {matches} null-like strings remaining")

In [ ]:
# Convert the min_taken_date back to string format, as it needs to display 00:00:00
df_merged['min_taken_date'] = pd.to_datetime(df_merged['min_taken_date']).dt.strftime('%Y-%m-%d %H:%M:%S')
# 10e. Confirm date columns are not corrupted
print(df_merged[['Post_date', 'Taken_date', 'min_taken_date']].head(10).to_string())


11. Rename columns to match sample output format

In [ ]:

# The sample output file specifies exact column names with underscores and
# title-case formatting. Renaming to match exactly.
column_rename_map = {
    'PostID':         'Post_ID',
    'UserID':         'User_ID',
    'secret':         'Secret',
    'server':         'Server',
    'title':          'Title',
    'ispublic':       'Is_Public',
    'isfriend':       'Is_Friend',
    'isfamily':       'Is_Family',
    'farm':           'Farm',
    'City':           'City',
    'Country':        'Country',
    'Post_date':      'Post_Date',
    'Taken_date':     'Taken_Date',
    'tags':           'Tags',
    'latitude':       'Latitude',
    'longitude':      'Longitude',
    'description':    'Description',
    'min_taken_date': 'Min_Taken_Date'
}

df_merged = df_merged.rename(columns=column_rename_map)
print("Final columns:", df_merged.columns.tolist())

In [ ]:
# 12. Export to csv
# Output with UTF-8 encoding and NaN representation as required by the spec
df_merged.to_csv('Group64_dataset.csv', index=False, encoding='utf-8', na_rep='NaN')
df_merged.shape

## 2. EDA


In [ ]:
# load package
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
# read the csv file
df = pd.read_csv("Group64_dataset.csv")

### 2.1 Dataset overview

In [ ]:
print(df.shape)
print("User:", df['User_ID'].nunique())
print("Country:", df['Country'].nunique())
print("City:", df['City'].nunique())
df.head(3)

In [ ]:
print('=== Data Types ===')
print(df.dtypes)

print('=== Descriptive Statistics ===')
df.describe()

### 2.2 Univariate analysis

2.2.1 Missing Value


In [ ]:
# missing value
missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(missing_pct.index, missing_pct.values,
               color=sns.color_palette('Reds_r', len(missing_pct)))
ax.set_xlabel('Missing Percentage (%)')
ax.set_title('Missing Value Rate per Column', fontweight='bold')
for bar, val in zip(bars, missing_pct.values):
    ax.text(val + 0.5,
            bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

plt.grid(False)
plt.savefig('fig1_missing_values.png', bbox_inches='tight')
plt.show()

2.2.2 Temporal Analysis

In [ ]:
# Post time analysis
# Parse dates
df['Post_Date']  = pd.to_datetime(df['Post_Date'],  errors='coerce')
df['Taken_Date'] = pd.to_datetime(df['Taken_Date'], errors='coerce')

# Reorder Monthly
df['year_month'] = df['Post_Date'].dt.to_period('M')
monthly_counts = df.groupby('year_month').size()
monthly_counts.index = monthly_counts.index.to_timestamp()

# Monthly-post plot
plt.figure(figsize=(10,4))
plt.grid(False)
plt.plot(monthly_counts.index, monthly_counts.values)

plt.title('Yearly Post Volume Over Time', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Number of Posts')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('fig3a_yearly_temporal_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# Weekly post analysis
df['post_week']  = df['Post_Date'].dt.dayofweek   # 0=Mon
dow_names   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(9, 4))

# Day of Week
dow_counts = df['post_week'].value_counts().sort_index()
colors_dow = ['#C44E52' if d == 6 else '#4C72B0' for d in dow_counts.index]
ax.bar([dow_names[i] for i in dow_counts.index],
            dow_counts.values, color=colors_dow, edgecolor='white')
ax.set_title('Posts by Day of Week', fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Number of Posts')

plt.grid(False)
plt.tight_layout()
plt.savefig('fig3c_weekly_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# Taken time vs Post time
df['taken_hour'] = df['Taken_Date'].dt.hour
df['post_hour']  = df['Post_Date'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Hourly post
hour_counts = df['post_hour'].value_counts().sort_index()
axes[0].bar(hour_counts.index, hour_counts.values,
           color='#4C72B0', edgecolor='white')
axes[0].set_title('Posts by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (24h)')
axes[0].set_ylabel('Number of Posts')
axes[0].legend(fontsize=9)

# ── Hourly taken
taken_counts = df['taken_hour'].value_counts().sort_index()
axes[1].bar(taken_counts.index, taken_counts.values,
            color="#CC5348", edgecolor='white')
axes[1].set_title('Taken by Hour of Day', fontweight='bold')
axes[1].set_xlabel('Hour (24h)')
axes[1].set_ylabel('Number of Taken')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig3b_postandtaken_analysis.png', bbox_inches='tight')
plt.show()

2.2.3 Geographic Distribution

In [ ]:
# Top20 Countries
fig, ax = plt.subplots(figsize=(15, 6))

country_counts = df['Country'].str.lower().str.strip().value_counts().head(20)
bars = ax.barh(country_counts.index[::-1], country_counts.values[::-1],
             color=sns.color_palette('Blues_d', 20))
ax.set_title('Top 20 Countries', fontweight='bold')
ax.set_xlabel('Number of Posts')

# add amount in the end
for bar in bars:
    bar_value = bar.get_width()

    ax.text(
        bar_value + 60,
        bar.get_y() + bar.get_height()/2,
        f'{int(bar_value)}',
        va='center',
        ha='left',
        fontsize=10
    )

plt.grid(False)
plt.tight_layout()
plt.savefig('fig5a_top20_countries_geographic.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top10 Cities
fig, ax = plt.subplots(figsize=(12, 4))

city_counts = df['City'].str.lower().str.strip().value_counts().head(10)
bars = ax.barh(city_counts.index[::-1], city_counts.values[::-1],
             color=sns.color_palette('Reds_d', 10))
ax.set_title('Top 10 Cities', fontweight='bold')
ax.set_xlabel('Number of Posts')

# add amount in the end
for bar in bars:
    bar_value = bar.get_width()

    ax.text(
        bar_value + 20,
        bar.get_y() + bar.get_height()/2,
        f'{int(bar_value)}',
        va='center',
        ha='left',
        fontsize=10
    )

plt.grid(False)
plt.tight_layout()
plt.savefig('fig5b_top10_cities_geographic.png', bbox_inches='tight')
plt.show()

2.2.4 Tag Analysis

In [ ]:
# 6. Tags Analysis — User Engagement via Tagging
df_tags = df[df['Tags'].notna()].copy()
df_tags['tag_list'] = df_tags['Tags'].str.strip('[]').str.replace("'", "").str.split(',')
df_tags['tag_count'] = df_tags['tag_list'].str.len()
avg_tags = df_tags['tag_count'].mean()
print(f'Average tags per post: {avg_tags:.2f}')

# All tags flattened
all_tags = [t.strip() for sublist in df_tags['tag_list']
            for t in sublist if t.strip()]
tag_freq = Counter(all_tags).most_common(20)
tag_words, tag_vals = zip(*tag_freq)

fig, ax = plt.subplots(figsize=(15, 5))

# Tag count distribution
ax.hist(df_tags['tag_count'].clip(upper=40), bins=40,
             color='#C44E52', edgecolor='white')
ax.axvline(df_tags['tag_count'].median(), color='navy',
                linestyle='--', label=f'Median = {df_tags["tag_count"].median():.0f} tags')
ax.set_title('Number of Tags per Post', fontweight='bold')
ax.set_xlabel('Tag Count')
ax.set_ylabel('Number of Posts')
ax.legend()

plt.grid(False)
plt.tight_layout()
plt.savefig('fig5c_geo_scatter.png', bbox_inches='tight')
plt.show()


In [ ]:
# Top 20 tags
fig, ax = plt.subplots(figsize=(15, 7))

bars = ax.barh(list(tag_words)[::-1], list(tag_vals)[::-1],
             color=sns.color_palette('Blues_d', 20))
ax.set_title('Top 20 Most Used Tags', fontweight='bold')
ax.set_xlabel('Frequency')

# add amount in the end
for bar in bars:
    bar_value = bar.get_width()

    ax.text(
        bar_value + 30,
        bar.get_y() + bar.get_height()/2,
        f'{int(bar_value)}',
        va='center',
        ha='left',
        fontsize=10
    )

plt.tight_layout()
plt.savefig('fig6_tags_analysis.png', bbox_inches='tight')
plt.show()

2.2.5 Description Length analysis

In [ ]:
# split the str in description
df_desc = df[df['Description'].notna()].copy()
df_desc['desc_len'] = df_desc['Description'].str.len()

# Categorise
bins   = [0, 50, 150, 500, np.inf]
labels = ['Very Short\n(<50)', 'Short\n(50–150)', 'Medium\n(150–500)', 'Long\n(500+)']
df_desc['desc_cat'] = pd.cut(df_desc['desc_len'], bins=bins, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(df_desc['desc_len'].clip(upper=1000), bins=50,
             color="#ED4B0BC7", edgecolor='white')
axes[0].axvline(df_desc['desc_len'].median(), color='red',
                linestyle='--',
                label=f'Median = {df_desc["desc_len"].median():.0f} chars')
axes[0].set_title('Description Length Distribution', fontweight='bold')
axes[0].set_xlabel('Characters (clipped at 1000)')
axes[0].set_ylabel('Posts')
axes[0].legend()

# Category bar
cat_counts = df_desc['desc_cat'].value_counts().reindex(labels)
axes[1].bar(cat_counts.index, cat_counts.values,
            color=['#4C72B0','#DD8452','#55A868','#C44E52'], edgecolor='white')
axes[1].set_title('Description Length Categories', fontweight='bold')
axes[1].set_ylabel('Number of Posts')
for i, v in enumerate(cat_counts.values):
    axes[1].text(i, v + 50, f'{int(v):,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig7_description_length.png', bbox_inches='tight')
plt.show()
print('Finding: 57% of posts with descriptions are "Very Short" (<50 chars),'
      ' showing most users prefer minimal text. Only 14% write long descriptions (500+ chars).')

### 2.3 Bivariate analysis

2.3.1 Upload delay analysis

In [ ]:
# upload delay from taken to post
df['post_dow'] = df['Post_Date'].dt.dayofweek
df['upload_delay_h'] = (df['Post_Date'] - df['Taken_Date']).dt.total_seconds() / 3600
delay_clean = df['upload_delay_h'][(df['upload_delay_h'] >= 0) &
                                    (df['upload_delay_h'] <= 720)]  # within 30 days

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(delay_clean, bins=50, color='#DD8452', edgecolor='white')
axes[0].set_title('Upload Delay Distribution (0–720h)', fontweight='bold')
axes[0].set_xlabel('Delay (hours)')
axes[0].set_ylabel('Number of Posts')
axes[0].axvline(delay_clean.median(), color='red',
                linestyle='--', label=f'Median = {delay_clean.median():.1f}h')
axes[0].legend()

print(df['upload_delay_h'].mean())
df['upload_delay_h'].describe()
median = delay_clean.median()
print(median)

# Box plot by day of week
delay_df = df[['post_dow', 'upload_delay_h']].copy()
delay_df = delay_df[(delay_df['upload_delay_h'] >= 0) &
                    (delay_df['upload_delay_h'] <= 720)]
delay_df['Day'] = delay_df['post_dow'].map(dict(enumerate(dow_names)))
order = dow_names
sns.boxplot(data=delay_df, x='Day', y='upload_delay_h',
            order=order, ax=axes[1], palette='muted', fliersize=2)
axes[1].set_title('Upload Delay by Day of Week', fontweight='bold')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Delay (hours)')

plt.tight_layout()
plt.savefig('fig4_upload_delay.png', bbox_inches='tight')
plt.show()

2.3.2  Location map

In [ ]:
# europe map
import geopandas as gpd
# ── Geospatial scatter plot
geo = df[df['Latitude'].notna() & df['Longitude'].notna()].copy()
fig, ax = plt.subplots(figsize=(20, 6))

# Google Maps-style colors
ax.set_facecolor("#e8f0e8")          # water/background = light blue-green

world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)
europe = world[world["REGION_UN"] == "Europe"]
europe.plot(
    ax=ax,
    color="#f5f5dc",        # land = warm beige (Google Maps land color)
    edgecolor="#c0c0c0",    # borders = soft gray
    linewidth=0.5
)

# Original la&long titude
scatter = ax.scatter(geo['Longitude'], geo['Latitude'],
                     alpha=0.15, s=5, c='#4C72B0')
ax.set_title('Geospatial Distribution of Posts', fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_xlim(-12, 12)
ax.set_ylim(50, 62)

mean_lon = df['Longitude'].mean()
mean_lat = df['Latitude'].mean()

plt.axvline(mean_lon, linestyle='--')
plt.axhline(mean_lat, linestyle='--')

plt.grid(False)
plt.tight_layout()
plt.savefig('fig_geo_scatter.png', bbox_inches='tight')
plt.show()

### 2.4 Multivariate analysis

In [ ]:
# Correlation Heatmap — Numerical Features
# Add engineered features for correlation
df['has_tags']        = df['Tags'].notna().astype(int)
df['has_description'] = df['Description'].notna().astype(int)
df['has_city']        = df['City'].notna().astype(int)
df['has_country']     = df['Country'].notna().astype(int)
# df['tag_count']       = df['Tags'].str.strip(',').str.split(',').str.len().fillna(0)
df['tag_count'] = (
    df['Tags']
    .fillna('')
    .str.strip(',')
    .str.split(',')
    .apply(lambda x: 0 if x == [''] else len(x))
)

corr_cols = [
    'post_hour', 'tag_count',
    'has_country', 'has_city',
    'has_description', 'has_tags']

corr = df[corr_cols].corr()
df_corr = df.drop(columns=['post_dow'])

fig, ax = plt.subplots(figsize=(13, 5))

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            ax=ax, annot_kws={'size': 9})
ax.set_title('Correlation Heatmap of Key Features', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 3.Key insights and research questions

### 3.1 key findings

### 3.2 Machine Learning research questions and justification

# Reference